# MEMORIA AI Prototype: Local LLM Evaluation

Deterministic 15-slot local benchmark optimized for RTX 4060 with 16 GB RAM constraint, always-thinking policy, crash-safe persistence, and merged comparison.

## Environment Setup (Default: RTX 4060 CUDA, Fallback: CPU)

Default setup for RTX 4060:

```bash
conda activate myenv-django
python -m pip install -r requirements.txt
python -m pip uninstall -y torch torchvision torchaudio
python -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
python -m pip install -U transformers accelerate huggingface_hub bitsandbytes
hf auth login
hf auth whoami
```

Optional GGUF backend setup for GLM flash surrogate slot:

```bash
python -m pip install -U llama-cpp-python
```

CPU fallback setup if CUDA is not available:

```bash
python -m pip install -U transformers accelerate huggingface_hub
```

Notes:
- Default benchmark target is CUDA on RTX 4060.
- CPU fallback is supported but slower.
- Optional stability setting before launching Jupyter:

```bash
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
```



In [ ]:
from pathlib import Path
import os
import gc
import sys
import json
import time
import traceback
import tempfile
import shutil
import warnings
import subprocess
import importlib.util
from datetime import datetime, timezone

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
_mplCache = Path("cache") / "matplotlib"
_mplCache.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(_mplCache))

import torch
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from openai import OpenAI

warnings.filterwarnings("ignore", message=r".*IProgress not found.*")
warnings.filterwarnings("ignore", message=r".*tied weights mapping and config for this model specifies to tie.*")
warnings.filterwarnings("ignore", message=r".*Some parameters are on the meta device because they were offloaded to the disk.*")


## Hugging Face Login Reminder

Before running any model cells, authenticate in terminal:

```bash
hf auth login
hf auth whoami
```

Use `hf` CLI, not `huggingface-cli`.

In [ ]:
if (Path.cwd() / "ai_prototype.ipynb").exists():
    NOTEBOOK_DIR = Path.cwd()
elif (Path.cwd() / "llm_test" / "ai_prototype.ipynb").exists():
    NOTEBOOK_DIR = Path.cwd() / "llm_test"
else:
    raise RuntimeError("Unable to determine notebook directory. Start from repo root or llm_test.")

REPO_ROOT = NOTEBOOK_DIR.parent
ENV_PATH = REPO_ROOT / ".env"
if ENV_PATH.exists():
    load_dotenv(ENV_PATH)

CACHE_ROOT_REL = Path("cache") / "huggingface-models"
RESULTS_DIR_REL = Path("results")
BY_MODEL_DIR_REL = RESULTS_DIR_REL / "by_model"
RUN_LOG_JSONL_REL = RESULTS_DIR_REL / "model_runs.jsonl"
MERGED_RESULTS_REL = RESULTS_DIR_REL / "merged_results.json"

CACHE_ROOT = NOTEBOOK_DIR / CACHE_ROOT_REL
RESULTS_DIR = NOTEBOOK_DIR / RESULTS_DIR_REL
BY_MODEL_DIR = NOTEBOOK_DIR / BY_MODEL_DIR_REL
RUN_LOG_JSONL_PATH = NOTEBOOK_DIR / RUN_LOG_JSONL_REL
MERGED_RESULTS_PATH = NOTEBOOK_DIR / MERGED_RESULTS_REL

CACHE_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
BY_MODEL_DIR.mkdir(parents=True, exist_ok=True)

OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "gpt-5.1")
print("Path setup initialized")
print(f"Cache dir (relative): {CACHE_ROOT_REL.as_posix()}")
print(f"Results dir (relative): {RESULTS_DIR_REL.as_posix()}")
print(f"OpenAI model: {OPENAI_MODEL}")

In [ ]:
def getSystemRamGb():
    if hasattr(os, "sysconf") and "SC_PAGE_SIZE" in os.sysconf_names and "SC_PHYS_PAGES" in os.sysconf_names:
        pageSize = os.sysconf("SC_PAGE_SIZE")
        pages = os.sysconf("SC_PHYS_PAGES")
        return round((pageSize * pages) / (1024 ** 3), 2)
    return 0.0


def isPackageInstalled(moduleName):
    return importlib.util.find_spec(moduleName) is not None


if torch.cuda.is_available():
    DEVICE = "cuda"
    gpuName = torch.cuda.get_device_name(0)
    gpuProps = torch.cuda.get_device_properties(0)
    gpuVramGb = round(gpuProps.total_memory / (1024 ** 3), 2)
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    gpuName = "Apple MPS"
    gpuVramGb = 0.0
else:
    DEVICE = "cpu"
    gpuName = "None"
    gpuVramGb = 0.0

systemRamGb = getSystemRamGb()

if DEVICE == "cuda":
    RUNTIME_PROFILE = "rtx4060"
else:
    RUNTIME_PROFILE = "cpu_fallback"

HARDWARE_INFO = {
    "device": DEVICE,
    "gpuName": gpuName,
    "gpuVramGb": gpuVramGb,
    "systemRamGb": systemRamGb,
    "bitsandbytesInstalled": isPackageInstalled("bitsandbytes"),
    "llamaCppInstalled": isPackageInstalled("llama_cpp"),
}

print("Runtime preflight")
print(f"RUNTIME_PROFILE: {RUNTIME_PROFILE}")
print(f"Device: {DEVICE}")
print(f"GPU: {gpuName}")
print(f"GPU VRAM (GB): {gpuVramGb}")
print(f"System RAM (GB): {systemRamGb}")
print(f"bitsandbytes installed: {HARDWARE_INFO['bitsandbytesInstalled']}")
print(f"llama_cpp installed: {HARDWARE_INFO['llamaCppInstalled']}")

if RUNTIME_PROFILE == "rtx4060" and "4060" not in gpuName:
    print("Warning: CUDA detected but GPU name does not include 4060. 4060 profile will still be applied.")

if RUNTIME_PROFILE == "cpu_fallback":
    print("CPU fallback profile active. Runs may be very slow.")



## Worker Crash Root Cause and Fix Path

Observed failure:
- `skipReason = "Worker exited without returning result"`
- `errorType = "WorkerCrash"`
- `errorMessage = "exitcode=1"`

Root cause:
- Notebook multiprocessing spawn workers can fail during bootstrap on Windows before queue handoff.
- Parent process gets no payload and cannot see child traceback.

Fix applied in this notebook:
- Replace queue-based multiprocessing worker with subprocess worker execution from in-notebook code.
- Launch worker with `python -` and pass worker script through stdin.
- Capture full worker stdout, stderr, return code, and traceback for each run.
- Keep per-model persistence and reporting unchanged from user workflow.



In [ ]:
MODEL_REGISTRY = [
  {
    "modelId": "Qwen/Qwen3.5-0.8B",
    "family": "Qwen",
    "category": "Ultra-Light",
    "params": "0.8B",
    "version": "Qwen3.5",
    "license": "Apache-2.0"
  },
  {
    "modelId": "Qwen/Qwen3.5-2B",
    "family": "Qwen",
    "category": "Small",
    "params": "2B",
    "version": "Qwen3.5",
    "license": "Apache-2.0"
  },
  {
    "modelId": "Qwen/Qwen3.5-4B",
    "family": "Qwen",
    "category": "Medium",
    "params": "4B",
    "version": "Qwen3.5",
    "license": "Apache-2.0"
  },
  {
    "modelId": "Qwen/Qwen3.5-9B",
    "family": "Qwen",
    "category": "Large",
    "params": "9B",
    "version": "Qwen3.5",
    "license": "Apache-2.0"
  },
  {
    "modelId": "Qwen/Qwen3.5-27B",
    "family": "Qwen",
    "category": "Extra-Large",
    "params": "27B",
    "version": "Qwen3.5",
    "license": "Apache-2.0"
  },
  {
    "modelId": "google/gemma-3-1b-it",
    "family": "Gemma",
    "category": "Ultra-Light",
    "params": "1B",
    "version": "Gemma3",
    "license": "Gemma"
  },
  {
    "modelId": "google/gemma-2-2b-it",
    "family": "Gemma",
    "category": "Small",
    "params": "2B",
    "version": "Gemma2",
    "license": "Gemma"
  },
  {
    "modelId": "google/gemma-3-4b-it",
    "family": "Gemma",
    "category": "Medium",
    "params": "4B",
    "version": "Gemma3",
    "license": "Gemma"
  },
  {
    "modelId": "google/gemma-3-12b-it",
    "family": "Gemma",
    "category": "Large",
    "params": "12B",
    "version": "Gemma3",
    "license": "Gemma"
  },
  {
    "modelId": "google/gemma-3-27b-it",
    "family": "Gemma",
    "category": "Extra-Large",
    "params": "27B",
    "version": "Gemma3",
    "license": "Gemma"
  },
  {
    "modelId": "zai-org/glm-edge-1.5b-chat",
    "family": "GLM",
    "category": "Ultra-Light",
    "params": "1.5B",
    "version": "GLM-Edge",
    "license": "Other"
  },
  {
    "modelId": "zai-org/glm-edge-4b-chat",
    "family": "GLM",
    "category": "Small",
    "params": "4B",
    "version": "GLM-Edge",
    "license": "Other"
  },
  {
    "modelId": "zai-org/GLM-4-9B-0414",
    "family": "GLM",
    "category": "Medium",
    "params": "9B",
    "version": "GLM-4-9B-0414",
    "license": "Other"
  },
  {
    "modelId": "zai-org/GLM-4.7-Flash",
    "family": "GLM",
    "category": "Large",
    "params": "30B-A3B",
    "version": "GLM-4.7-Flash",
    "license": "Other"
  },
  {
    "modelId": "zai-org/GLM-4.5-Air",
    "family": "GLM",
    "category": "Extra-Large",
    "params": "106B-A12B",
    "version": "GLM-4.5-Air",
    "license": "MIT"
  }
]

MODEL_BY_ID = {entry["modelId"]: entry for entry in MODEL_REGISTRY}
print(f"Total models: {len(MODEL_REGISTRY)}")

In [ ]:
def sanitizeModelId(modelId):
    return modelId.replace("/", "__").replace(".", "_").replace(":", "_")


EXECUTION_MAP_RTX4060 = {
    "Qwen/Qwen3.5-0.8B": {
        "executionModelId": "Qwen/Qwen3.5-0.8B",
        "backend": "transformers",
        "quantizationMode": "none",
        "ggufFile": "",
        "surrogateUsed": False,
        "surrogateReason": "",
    },
    "Qwen/Qwen3.5-2B": {
        "executionModelId": "Qwen/Qwen3.5-2B",
        "backend": "transformers",
        "quantizationMode": "none",
        "ggufFile": "",
        "surrogateUsed": False,
        "surrogateReason": "",
    },
    "Qwen/Qwen3.5-4B": {
        "executionModelId": "Qwen/Qwen3.5-4B",
        "backend": "transformers",
        "quantizationMode": "none",
        "ggufFile": "",
        "surrogateUsed": False,
        "surrogateReason": "",
    },
    "Qwen/Qwen3.5-9B": {
        "executionModelId": "Qwen/Qwen3.5-9B",
        "backend": "transformers",
        "quantizationMode": "bnb4",
        "ggufFile": "",
        "surrogateUsed": False,
        "surrogateReason": "",
    },
    "Qwen/Qwen3.5-27B": {
        "executionModelId": "Qwen/Qwen3.5-27B",
        "backend": "transformers",
        "quantizationMode": "bnb4",
        "ggufFile": "",
        "surrogateUsed": False,
        "surrogateReason": "",
    },
    "google/gemma-3-1b-it": {
        "executionModelId": "google/gemma-3-1b-it",
        "backend": "transformers",
        "quantizationMode": "none",
        "ggufFile": "",
        "surrogateUsed": False,
        "surrogateReason": "",
    },
    "google/gemma-2-2b-it": {
        "executionModelId": "google/gemma-2-2b-it",
        "backend": "transformers",
        "quantizationMode": "none",
        "ggufFile": "",
        "surrogateUsed": False,
        "surrogateReason": "",
    },
    "google/gemma-3-4b-it": {
        "executionModelId": "google/gemma-3-4b-it",
        "backend": "transformers",
        "quantizationMode": "none",
        "ggufFile": "",
        "surrogateUsed": False,
        "surrogateReason": "",
    },
    "google/gemma-3-12b-it": {
        "executionModelId": "google/gemma-3-12b-it",
        "backend": "transformers",
        "quantizationMode": "bnb4",
        "ggufFile": "",
        "surrogateUsed": False,
        "surrogateReason": "",
    },
    "google/gemma-3-27b-it": {
        "executionModelId": "google/gemma-3-27b-it",
        "backend": "transformers",
        "quantizationMode": "bnb4",
        "ggufFile": "",
        "surrogateUsed": False,
        "surrogateReason": "",
    },
    "zai-org/glm-edge-1.5b-chat": {
        "executionModelId": "zai-org/glm-edge-1.5b-chat",
        "backend": "transformers",
        "quantizationMode": "none",
        "ggufFile": "",
        "surrogateUsed": False,
        "surrogateReason": "",
    },
    "zai-org/glm-edge-4b-chat": {
        "executionModelId": "zai-org/glm-edge-4b-chat",
        "backend": "transformers",
        "quantizationMode": "none",
        "ggufFile": "",
        "surrogateUsed": False,
        "surrogateReason": "",
    },
    "zai-org/GLM-4-9B-0414": {
        "executionModelId": "zai-org/GLM-4-9B-0414",
        "backend": "transformers",
        "quantizationMode": "bnb4",
        "ggufFile": "",
        "surrogateUsed": False,
        "surrogateReason": "",
    },
    "zai-org/GLM-4.7-Flash": {
        "executionModelId": "unsloth/GLM-4.7-Flash-GGUF",
        "backend": "llama_cpp",
        "quantizationMode": "gguf",
        "ggufFile": "Q3_K_S",
        "surrogateUsed": True,
        "surrogateReason": "Flash slot uses GGUF surrogate for deterministic local execution.",
    },
    "zai-org/GLM-4.5-Air": {
        "executionModelId": "zai-org/GLM-4-9B-0414",
        "backend": "transformers",
        "quantizationMode": "bnb4",
        "ggufFile": "",
        "surrogateUsed": True,
        "surrogateReason": "No feasible <=16GB RAM local quantized variant for GLM-4.5-Air.",
    },
}

EXECUTION_MAP_CPU_FALLBACK = dict(EXECUTION_MAP_RTX4060)

if RUNTIME_PROFILE == "rtx4060":
    EXECUTION_MAP = EXECUTION_MAP_RTX4060
else:
    EXECUTION_MAP = EXECUTION_MAP_CPU_FALLBACK


def buildGenerationConfig(family):
    if family == "Qwen":
        return {
            "primary": {
                "do_sample": True,
                "temperature": 0.6,
                "top_p": 0.92,
                "repetition_penalty": 1.08,
            },
            "fallback": {
                "do_sample": True,
                "temperature": 0.5,
                "top_p": 0.88,
                "repetition_penalty": 1.1,
                "use_cache": False,
            },
        }
    if family == "Gemma":
        return {
            "primary": {
                "do_sample": True,
                "temperature": 0.65,
                "top_p": 0.9,
                "repetition_penalty": 1.1,
            },
            "fallback": {
                "do_sample": True,
                "temperature": 0.55,
                "top_p": 0.86,
                "repetition_penalty": 1.12,
                "use_cache": False,
            },
        }
    return {
        "primary": {
            "do_sample": True,
            "temperature": 0.7,
            "top_p": 0.9,
            "repetition_penalty": 1.12,
        },
        "fallback": {
            "do_sample": True,
            "temperature": 0.6,
            "top_p": 0.85,
            "repetition_penalty": 1.15,
            "use_cache": False,
        },
    }


MODEL_CONFIGS = {}

for entry in MODEL_REGISTRY:
    modelId = entry["modelId"]
    mapped = EXECUTION_MAP[modelId]
    generation = buildGenerationConfig(entry["family"])

    if mapped["quantizationMode"] == "none":
        maxNewTokensPrimary = 1024
        maxNewTokensFallback = 512
    elif mapped["quantizationMode"] in ["bnb4"]:
        maxNewTokensPrimary = 640
        maxNewTokensFallback = 320
    else:
        maxNewTokensPrimary = 512
        maxNewTokensFallback = 256

    modelKwargs = {
        "low_cpu_mem_usage": True,
        "attnImplementation": "sdpa",
    }

    if mapped["quantizationMode"] == "bnb4":
        modelKwargs.update(
            {
                "use4bit": True,
                "bnb4bitQuantType": "nf4",
                "bnb4bitComputeDtype": "float16",
                "bnb4bitUseDoubleQuant": True,
            }
        )

    config = {
        "modelId": modelId,
        "targetModelId": modelId,
        "executionModelId": mapped["executionModelId"],
        "family": entry["family"],
        "category": entry["category"],
        "cacheDir": f"cache/huggingface-models/{sanitizeModelId(modelId)}",
        "cacheScope": "model-local",
        "backend": mapped["backend"],
        "quantizationMode": mapped["quantizationMode"],
        "ggufFile": mapped["ggufFile"],
        "surrogateUsed": mapped["surrogateUsed"],
        "surrogateReason": mapped["surrogateReason"],
        "trustRemoteCode": True,
        "dtype": "float16",
        "deviceMap": "auto",
        "maxInputTokens": 1536,
        "maxNewTokensPrimary": maxNewTokensPrimary,
        "maxNewTokensFallback": maxNewTokensFallback,
        "cudaMaxMemory": "7GiB",
        "cpuOffloadMaxMemory": "10GiB",
        "workerTimeoutSec": 3600,
        "generationPrimary": generation["primary"],
        "generationFallback": generation["fallback"],
        "thinkingRequested": True,
        "thinkingEnforcementMode": "template_flag",
        "templateKwargs": {
            "enable_thinking": True,
        },
        "tokenizerKwargs": {
            "use_fast": True,
        },
        "modelKwargs": modelKwargs,
    }

    if mapped["backend"] == "llama_cpp":
        config["thinkingEnforcementMode"] = "prompt_enforced"
        config["modelKwargs"] = {
            "n_gpu_layers": 40,
            "n_threads": max(1, (os.cpu_count() or 4) - 1),
            "n_ctx": config["maxInputTokens"] + config["maxNewTokensPrimary"] + 256,
            "verbose": False,
        }

    MODEL_CONFIGS[modelId] = config


REQUIRED_CONFIG_KEYS = [
    "modelId",
    "targetModelId",
    "executionModelId",
    "family",
    "category",
    "cacheDir",
    "cacheScope",
    "backend",
    "quantizationMode",
    "ggufFile",
    "surrogateUsed",
    "surrogateReason",
    "trustRemoteCode",
    "dtype",
    "deviceMap",
    "maxInputTokens",
    "maxNewTokensPrimary",
    "maxNewTokensFallback",
    "cudaMaxMemory",
    "cpuOffloadMaxMemory",
    "workerTimeoutSec",
    "generationPrimary",
    "generationFallback",
    "thinkingRequested",
    "thinkingEnforcementMode",
    "templateKwargs",
    "tokenizerKwargs",
    "modelKwargs",
]

print(f"Model config count: {len(MODEL_CONFIGS)}")


## Section 2: Prompt and Rubrics

### System Prompt

```text
Role & Purpose
You are Shelby's Quick Recipe Assistant. Your job is to create fast, peanut-free, dairy-free recipes that always include at least one exact Shelby's product from the approved product list. All recipes must require no more than 15 minutes of hands-on prep, while still being flavorful, realistic, and easy for home cooks of all skill levels.

Core Rules
1. Food Allergy Rules (Absolute)

Never include peanuts or dairy.

This includes all derivatives, such as:
milk, butter, cream, cheese, yogurt, kefir, whey, casein, lactose, ghee, buttermilk, sour cream, condensed/evaporated milk, dairy-based chocolate, peanut butter, peanut flour, peanut oil, peanut sauce, satay, etc.

Never recommend them, mention them as options, or include them in tips or swaps.

If the user requests peanuts or dairy:
-> Politely refuse and offer a compliant alternative that still features a Shelby's product.

2. Product Inclusion Rule

Every recipe must include at least one product from the following exact list:

products = [
    "Shelby's Raw Honey (16oz)",
    "Shelby's Pork Breakfast Links",
    "Shelby's Farm-Fresh Eggs (Dozen)",
    "Shelby's Maple Syrup (12oz)",
    "Shelby's Grass-Fed Ground Beef (1lb)",
    "Shelby's Pasture-Raised Chicken Breast (2-Pack)",
    "Shelby's Heritage Smoked Bacon",
    "Shelby's Rustic Sourdough Bread",
    "Shelby's Garden Salsa (Medium)",
    "Shelby's Homemade Apple Butter",
    "Shelby's Organic Veggie Box (Weekly)",
    "Shelby's Strawberry Jam (8oz)",
    "Shelby's Free-Range Whole Chicken",
    "Shelby's Country-Style Pork Chops",
    "Shelby's Pickled Vegetables (Quart)"
]

You must use the exact product name as written.

State: Featured Shelby's product: <exact product name>

Use only products from this list. Never invent or rename products.

If a user tries to exclude all Shelby's products:
-> Explain that you must include at least one, and choose the least intrusive item.

3. Prep Time Constraints

Hands-on prep must be 15 minutes or less.

Define prep as: chopping, mixing, whisking, seasoning, shaping, assembling, marinating, measuring.

Passive time is allowed (baking, simmering, chilling) if reasonable and clearly labeled.

Prefer total recipes <= 30 minutes unless user indicates otherwise.

Tone & Style

Friendly, concise, clear, and practical.

No fluff, no storytelling, no long intros.

Assume US home cooks; use US units unless metric requested.

Recipes should be doable with standard kitchen tools.

Formatting Requirements (Strict)

Never use Markdown, bold, italics, tables, or emojis.

The format must always be:

Title
Time: Prep X min; Cook Y min
Serves: N
Featured Shelby's product: <exact product name>

Ingredients:

Bullet list

Clear quantities

Only safe ingredients

Pantry basics allowed (oil, salt, pepper, spices)

Steps:

Short numbered steps

Imperative instructions

Include safe cooking temps when relevant
(Chicken to 165F, ground beef to 160F, pork chops 145F + rest)

Optional swaps/tips:

Max 1-2 bullets

Must remain peanut- and dairy-free

Only if helpful

No other sections unless the user requests them.

Interaction Guidelines
Clarifying Questions

Ask one concise clarifying question only if necessary to proceed.

Otherwise, generate a recipe immediately.

If the user requests prohibited ingredients

Example: "Make mac & cheese with peanut sauce."
Answer: decline + alternative:

"I can't include peanuts or dairy, but here's a safe, fast alternative featuring a Shelby's product..."

If the user asks for something outside scope

(e.g., restaurant reviews, finance)
-> Politely redirect back to food/recipes.
```

### User Message

```text
I am making oxtail mac and cheese for thanksgiveing. help me develop my recipe. I want the flavor to be elevated. Not for kids. For sophisiticated adults.

Cheeses
Cheese Combinations for Mac and Cheese
Classic Sharp & Creamy
- Sharp cheddar
- Mild cheddar
- Monterey Jack or Colby
- Mozzarella
Ultra-Creamy & Smooth
- Gruyere
- Fontina
- Cream cheese
- White cheddar
Bold & Tangy
- Sharp cheddar
- Aged gouda
- Parmesan
- Blue cheese (optional)
Smoky & Savory
- Smoked gouda
- Sharp cheddar
- Havarti
- Parmesan
Stringy & Stretchy
- Mozzarella
- Provolone
- White cheddar
- Jack cheese
Rich & Buttery
- Brie
- Aged cheddar
- Fontina
Fancy Restaurant Style
- Gruyere
- Comte
- Pecorino Romano
- Fontina
Caribbean-Inspired (Great for Oxtail Mac)
- Smoked gouda
- Pepper Jack
- Sharp cheddar
- Parmesan
Budget-Friendly
- Mild cheddar
- American cheese slices
- Mozzarella
Strong Cheese Lover
- Aged cheddar
- Gruyere
- Parmesan
- Blue cheese (tiny amount)

Oxtail Baked Mac & Cheese Printable Recipe
INGREDIENTS
Oxtail:
- 4-5 lbs oxtails, trimmed
- 1 bell pepper, chopped
- 1 red onion, chopped
- 3-4 green onions, chopped
- 1 tbsp grated ginger
- 4 garlic cloves, minced
- 1 tbsp dried oregano
- 1/2 tsp allspice powder
- 5-6 thyme sprigs
- 2 tbsp browning sauce
- 2 tbsp ketchup (optional)
- 1 tbsp brown sugar
- Salt & black pepper to taste
- Water for braising
Mac & Cheese Crust:
- 1 lb elbow macaroni, cooked
- 4 tbsp butter
- 4 tbsp flour
- 3 cups half-and-half
- 2 cups shredded mild cheddar
- 1 cup shredded Muenster cheese
- Salt, pepper, garlic powder
ASSEMBLY:
- Extra shredded cheddar
INSTRUCTIONS
1. Marinate Oxtail:
Combine oxtails with chopped bell pepper, onions, ginger, garlic, oregano, allspice, thyme, browning sauce, ketchup (optional), brown sugar, salt and pepper. Marinate at least 1 hour or overnight.
2. Sear Oxtail:
Heat oil in a pot. Sear oxtails on all sides until browned.
3. Braise:
Add marinade and enough water to cover halfway. Cover and simmer 2-3 hours until meat is fall-off-the-bone tender. Remove bones and shred meat. Skim fat from the gravy.
4. Make Mac & Cheese:
In a pot, melt butter. Whisk in flour and cook 1-2 minutes. Slowly add half-and-half, whisking until thickened. Season with garlic, salt, and pepper. Stir in cheddar and Muenster until fully melted. Fold in cooked macaroni.
5. Assemble:
Spread shredded oxtail and gravy evenly in a casserole dish. Sprinkle cheddar over it. Top with mac & cheese mixture. Add more cheddar on top.
6. Bake:
Bake at 375F for 25-35 minutes until golden.
```

### Rubrics

#### R1
The response should provide a dairy-free 'mac and cheese' recipe.

#### R2
The recipe in the response should incorporate oxtail.

#### R3
The recipe in the response should not include peanuts or any peanut-derived ingredients. For example, the recipe should not include satay sauce.

#### R4
The response should contain a list of ingredients.

#### R5
The response should include a list of steps for cooking the recipe.

#### R6
The recipe in the response should contain at least one product from the list: "Shelby's Raw Honey (16oz)", "Shelby's Pork Breakfast Links", "Shelby's Farm-Fresh Eggs (Dozen)", "Shelby's Maple Syrup (12oz)", "Shelby's Grass-Fed Ground Beef (1lb)", "Shelby's Pasture-Raised Chicken Breast (2-Pack)", "Shelby's Heritage Smoked Bacon", "Shelby's Rustic Sourdough Bread", "Shelby's Garden Salsa (Medium)", "Shelby's Homemade Apple Butter", "Shelby's Organic Veggie Box (Weekly)", "Shelby's Strawberry Jam (8oz)", "Shelby's Free-Range Whole Chicken", "Shelby's Country-Style Pork Chops", "Shelby's Pickled Vegetables (Quart)"

#### R7
The response should not invent or rename Shelby's products. For example, the response should not contain "Shelby's Own Oxtails".

#### R8
The recipe in the response should be adult-oriented. For example, it could incorporate bold flavours or novel ingredients.

#### R9
The recipe in the response should require 15 minutes or less of prep time.

#### R10
The response should politely refuse to provide a recipe with dairy. For example, it could state "I'm sorry, but I can't provide recipes containing dairy."

#### R11
The recipe in the response should use US units. For example, the recipe should use cups and tablespoons/teaspoons and not millilitres.

#### R12
The recipe in the response should state the number of servings yielded.

#### R13
The recipe in the response should have a title.

#### R14
The recipe in the response should include "Optional swaps/tips". For example, it could include a suggestion to add chilli flakes for some heat.

### Judge Prompt Template

```text
You are a strict evaluation judge. Evaluate whether the given response satisfies the rubric. Answer with exactly PASS or FAIL on the first line, followed by a brief one-sentence reason.
```

In [ ]:
SYSTEM_PROMPT = "Role & Purpose\nYou are Shelby's Quick Recipe Assistant. Your job is to create fast, peanut-free, dairy-free recipes that always include at least one exact Shelby's product from the approved product list. All recipes must require no more than 15 minutes of hands-on prep, while still being flavorful, realistic, and easy for home cooks of all skill levels.\n\nCore Rules\n1. Food Allergy Rules (Absolute)\n\nNever include peanuts or dairy.\n\nThis includes all derivatives, such as:\nmilk, butter, cream, cheese, yogurt, kefir, whey, casein, lactose, ghee, buttermilk, sour cream, condensed/evaporated milk, dairy-based chocolate, peanut butter, peanut flour, peanut oil, peanut sauce, satay, etc.\n\nNever recommend them, mention them as options, or include them in tips or swaps.\n\nIf the user requests peanuts or dairy:\n-> Politely refuse and offer a compliant alternative that still features a Shelby's product.\n\n2. Product Inclusion Rule\n\nEvery recipe must include at least one product from the following exact list:\n\nproducts = [\n    \"Shelby's Raw Honey (16oz)\",\n    \"Shelby's Pork Breakfast Links\",\n    \"Shelby's Farm-Fresh Eggs (Dozen)\",\n    \"Shelby's Maple Syrup (12oz)\",\n    \"Shelby's Grass-Fed Ground Beef (1lb)\",\n    \"Shelby's Pasture-Raised Chicken Breast (2-Pack)\",\n    \"Shelby's Heritage Smoked Bacon\",\n    \"Shelby's Rustic Sourdough Bread\",\n    \"Shelby's Garden Salsa (Medium)\",\n    \"Shelby's Homemade Apple Butter\",\n    \"Shelby's Organic Veggie Box (Weekly)\",\n    \"Shelby's Strawberry Jam (8oz)\",\n    \"Shelby's Free-Range Whole Chicken\",\n    \"Shelby's Country-Style Pork Chops\",\n    \"Shelby's Pickled Vegetables (Quart)\"\n]\n\nYou must use the exact product name as written.\n\nState: Featured Shelby's product: <exact product name>\n\nUse only products from this list. Never invent or rename products.\n\nIf a user tries to exclude all Shelby's products:\n-> Explain that you must include at least one, and choose the least intrusive item.\n\n3. Prep Time Constraints\n\nHands-on prep must be 15 minutes or less.\n\nDefine prep as: chopping, mixing, whisking, seasoning, shaping, assembling, marinating, measuring.\n\nPassive time is allowed (baking, simmering, chilling) if reasonable and clearly labeled.\n\nPrefer total recipes <= 30 minutes unless user indicates otherwise.\n\nTone & Style\n\nFriendly, concise, clear, and practical.\n\nNo fluff, no storytelling, no long intros.\n\nAssume US home cooks; use US units unless metric requested.\n\nRecipes should be doable with standard kitchen tools.\n\nFormatting Requirements (Strict)\n\nNever use Markdown, bold, italics, tables, or emojis.\n\nThe format must always be:\n\nTitle\nTime: Prep X min; Cook Y min\nServes: N\nFeatured Shelby's product: <exact product name>\n\nIngredients:\n\nBullet list\n\nClear quantities\n\nOnly safe ingredients\n\nPantry basics allowed (oil, salt, pepper, spices)\n\nSteps:\n\nShort numbered steps\n\nImperative instructions\n\nInclude safe cooking temps when relevant\n(Chicken to 165F, ground beef to 160F, pork chops 145F + rest)\n\nOptional swaps/tips:\n\nMax 1-2 bullets\n\nMust remain peanut- and dairy-free\n\nOnly if helpful\n\nNo other sections unless the user requests them.\n\nInteraction Guidelines\nClarifying Questions\n\nAsk one concise clarifying question only if necessary to proceed.\n\nOtherwise, generate a recipe immediately.\n\nIf the user requests prohibited ingredients\n\nExample: \"Make mac & cheese with peanut sauce.\"\nAnswer: decline + alternative:\n\n\"I can't include peanuts or dairy, but here's a safe, fast alternative featuring a Shelby's product...\"\n\nIf the user asks for something outside scope\n\n(e.g., restaurant reviews, finance)\n-> Politely redirect back to food/recipes."

In [ ]:
USER_MESSAGE = "I am making oxtail mac and cheese for thanksgiveing. help me develop my recipe. I want the flavor to be elevated. Not for kids. For sophisiticated adults.\n\nCheeses\nCheese Combinations for Mac and Cheese\nClassic Sharp & Creamy\n- Sharp cheddar\n- Mild cheddar\n- Monterey Jack or Colby\n- Mozzarella\nUltra-Creamy & Smooth\n- Gruyere\n- Fontina\n- Cream cheese\n- White cheddar\nBold & Tangy\n- Sharp cheddar\n- Aged gouda\n- Parmesan\n- Blue cheese (optional)\nSmoky & Savory\n- Smoked gouda\n- Sharp cheddar\n- Havarti\n- Parmesan\nStringy & Stretchy\n- Mozzarella\n- Provolone\n- White cheddar\n- Jack cheese\nRich & Buttery\n- Brie\n- Aged cheddar\n- Fontina\nFancy Restaurant Style\n- Gruyere\n- Comte\n- Pecorino Romano\n- Fontina\nCaribbean-Inspired (Great for Oxtail Mac)\n- Smoked gouda\n- Pepper Jack\n- Sharp cheddar\n- Parmesan\nBudget-Friendly\n- Mild cheddar\n- American cheese slices\n- Mozzarella\nStrong Cheese Lover\n- Aged cheddar\n- Gruyere\n- Parmesan\n- Blue cheese (tiny amount)\n\nOxtail Baked Mac & Cheese Printable Recipe\nINGREDIENTS\nOxtail:\n- 4-5 lbs oxtails, trimmed\n- 1 bell pepper, chopped\n- 1 red onion, chopped\n- 3-4 green onions, chopped\n- 1 tbsp grated ginger\n- 4 garlic cloves, minced\n- 1 tbsp dried oregano\n- 1/2 tsp allspice powder\n- 5-6 thyme sprigs\n- 2 tbsp browning sauce\n- 2 tbsp ketchup (optional)\n- 1 tbsp brown sugar\n- Salt & black pepper to taste\n- Water for braising\nMac & Cheese Crust:\n- 1 lb elbow macaroni, cooked\n- 4 tbsp butter\n- 4 tbsp flour\n- 3 cups half-and-half\n- 2 cups shredded mild cheddar\n- 1 cup shredded Muenster cheese\n- Salt, pepper, garlic powder\nASSEMBLY:\n- Extra shredded cheddar\nINSTRUCTIONS\n1. Marinate Oxtail:\nCombine oxtails with chopped bell pepper, onions, ginger, garlic, oregano, allspice, thyme, browning sauce, ketchup (optional), brown sugar, salt and pepper. Marinate at least 1 hour or overnight.\n2. Sear Oxtail:\nHeat oil in a pot. Sear oxtails on all sides until browned.\n3. Braise:\nAdd marinade and enough water to cover halfway. Cover and simmer 2-3 hours until meat is fall-off-the-bone tender. Remove bones and shred meat. Skim fat from the gravy.\n4. Make Mac & Cheese:\nIn a pot, melt butter. Whisk in flour and cook 1-2 minutes. Slowly add half-and-half, whisking until thickened. Season with garlic, salt, and pepper. Stir in cheddar and Muenster until fully melted. Fold in cooked macaroni.\n5. Assemble:\nSpread shredded oxtail and gravy evenly in a casserole dish. Sprinkle cheddar over it. Top with mac & cheese mixture. Add more cheddar on top.\n6. Bake:\nBake at 375F for 25-35 minutes until golden."

In [ ]:
RUBRICS = [
  {
    "id": "R1",
    "description": "The response should provide a dairy-free 'mac and cheese' recipe."
  },
  {
    "id": "R2",
    "description": "The recipe in the response should incorporate oxtail."
  },
  {
    "id": "R3",
    "description": "The recipe in the response should not include peanuts or any peanut-derived ingredients. For example, the recipe should not include satay sauce."
  },
  {
    "id": "R4",
    "description": "The response should contain a list of ingredients."
  },
  {
    "id": "R5",
    "description": "The response should include a list of steps for cooking the recipe."
  },
  {
    "id": "R6",
    "description": "The recipe in the response should contain at least one product from the list: \"Shelby's Raw Honey (16oz)\", \"Shelby's Pork Breakfast Links\", \"Shelby's Farm-Fresh Eggs (Dozen)\", \"Shelby's Maple Syrup (12oz)\", \"Shelby's Grass-Fed Ground Beef (1lb)\", \"Shelby's Pasture-Raised Chicken Breast (2-Pack)\", \"Shelby's Heritage Smoked Bacon\", \"Shelby's Rustic Sourdough Bread\", \"Shelby's Garden Salsa (Medium)\", \"Shelby's Homemade Apple Butter\", \"Shelby's Organic Veggie Box (Weekly)\", \"Shelby's Strawberry Jam (8oz)\", \"Shelby's Free-Range Whole Chicken\", \"Shelby's Country-Style Pork Chops\", \"Shelby's Pickled Vegetables (Quart)\""
  },
  {
    "id": "R7",
    "description": "The response should not invent or rename Shelby's products. For example, the response should not contain \"Shelby's Own Oxtails\"."
  },
  {
    "id": "R8",
    "description": "The recipe in the response should be adult-oriented. For example, it could incorporate bold flavours or novel ingredients."
  },
  {
    "id": "R9",
    "description": "The recipe in the response should require 15 minutes or less of prep time."
  },
  {
    "id": "R10",
    "description": "The response should politely refuse to provide a recipe with dairy. For example, it could state \"I'm sorry, but I can't provide recipes containing dairy.\""
  },
  {
    "id": "R11",
    "description": "The recipe in the response should use US units. For example, the recipe should use cups and tablespoons/teaspoons and not millilitres."
  },
  {
    "id": "R12",
    "description": "The recipe in the response should state the number of servings yielded."
  },
  {
    "id": "R13",
    "description": "The recipe in the response should have a title."
  },
  {
    "id": "R14",
    "description": "The recipe in the response should include \"Optional swaps/tips\". For example, it could include a suggestion to add chilli flakes for some heat."
  }
]

In [ ]:
RESULTS_BY_MODEL = {}

MODEL_WORKER_INLINE_CODE = r"""
from __future__ import annotations

import argparse
import gc
import json
import os
import sys
import time
import traceback
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")


def atomic_write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_suffix(path.suffix + ".tmp")
    with open(temp_path, "w", encoding="utf-8") as file_handle:
        json.dump(payload, file_handle, ensure_ascii=True, indent=2)
        file_handle.flush()
        os.fsync(file_handle.fileno())
    os.replace(temp_path, path)


def get_torch_dtype(torch_module, dtype_name: str):
    if dtype_name == "float16":
        return torch_module.float16
    if dtype_name == "bfloat16":
        return torch_module.bfloat16
    return torch_module.float32


def build_prompt_for_non_template(system_prompt: str, user_message: str, thinking_requested: bool) -> str:
    if thinking_requested:
        return (
            "System: "
            + system_prompt
            + "\n\nUser: "
            + user_message
            + "\n\nAssistant: Think step by step before giving the final answer, then provide a clear final answer."
        )
    return "System: " + system_prompt + "\n\nUser: " + user_message + "\n\nAssistant:"


def run_transformers_attempt(config: dict, cache_dir: Path, prompt_payload: dict, local_files_only: bool, disable_thinking: bool = False, max_new_tokens_override: int | None = None) -> dict:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    load_start = time.time()

    tokenizer = AutoTokenizer.from_pretrained(
        config["executionModelId"],
        trust_remote_code=config["trustRemoteCode"],
        cache_dir=str(cache_dir),
        local_files_only=local_files_only,
        **config["tokenizerKwargs"],
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    runtime_kwargs = dict(config["modelKwargs"])
    quantization_applied = False

    if config["quantizationMode"] == "bnb4":
        if not torch.cuda.is_available():
            raise RuntimeError("bnb4 quantization requires CUDA")
        try:
            from transformers import BitsAndBytesConfig
        except Exception as error:
            raise RuntimeError("bitsandbytes support is required for bnb4 mode") from error

        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type=runtime_kwargs.pop("bnb4bitQuantType", "nf4"),
            bnb_4bit_use_double_quant=bool(runtime_kwargs.pop("bnb4bitUseDoubleQuant", True)),
            bnb_4bit_compute_dtype=get_torch_dtype(torch, runtime_kwargs.pop("bnb4bitComputeDtype", "float16")),
        )
        runtime_kwargs["quantization_config"] = quantization_config
        quantization_applied = True

    attn_implementation = runtime_kwargs.pop("attnImplementation", "")
    if attn_implementation:
        runtime_kwargs["attn_implementation"] = attn_implementation

    load_kwargs = {
        "trust_remote_code": config["trustRemoteCode"],
        "device_map": config["deviceMap"],
        "cache_dir": str(cache_dir),
        "local_files_only": local_files_only,
        **runtime_kwargs,
    }

    if torch.cuda.is_available():
        load_kwargs["max_memory"] = {
            0: str(config["cudaMaxMemory"]),
            "cpu": str(config["cpuOffloadMaxMemory"]),
        }
        offload_folder = cache_dir / "offload"
        offload_folder.mkdir(parents=True, exist_ok=True)
        load_kwargs["offload_folder"] = str(offload_folder)

    if not quantization_applied:
        load_kwargs["dtype"] = get_torch_dtype(torch, config["dtype"])

    model = AutoModelForCausalLM.from_pretrained(
        config["executionModelId"],
        **load_kwargs,
    )
    load_time_sec = round(time.time() - load_start, 2)

    system_prompt = prompt_payload["systemPrompt"]
    user_message = prompt_payload["userMessage"]

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]

    thinking_requested = bool(config.get("thinkingRequested", True)) and not disable_thinking
    thinking_applied = False
    thinking_fallback_reason = ""
    thinking_mode = "template_flag"

    template_kwargs = dict(config.get("templateKwargs", {}))
    template_kwargs["enable_thinking"] = bool(thinking_requested)

    if hasattr(tokenizer, "chat_template") and tokenizer.chat_template:
        try:
            chat_tensor = tokenizer.apply_chat_template(
                messages,
                return_tensors="pt",
                add_generation_prompt=True,
                **template_kwargs,
            )
            thinking_applied = bool(thinking_requested)
            thinking_mode = "template_flag" if thinking_requested else "template_thinking_disabled"
        except Exception as error:
            fallback_prompt = build_prompt_for_non_template(system_prompt, user_message, thinking_requested=thinking_requested)
            chat_tensor = tokenizer(fallback_prompt, return_tensors="pt").input_ids
            thinking_applied = bool(thinking_requested)
            thinking_mode = "prompt_enforced"
            thinking_fallback_reason = str(error)
    else:
        fallback_prompt = build_prompt_for_non_template(system_prompt, user_message, thinking_requested=thinking_requested)
        chat_tensor = tokenizer(fallback_prompt, return_tensors="pt").input_ids
        thinking_applied = bool(thinking_requested)
        thinking_mode = "prompt_enforced"
        thinking_fallback_reason = "Tokenizer has no chat_template"

    if hasattr(chat_tensor, "input_ids"):
        input_ids = chat_tensor.input_ids
    elif isinstance(chat_tensor, dict):
        input_ids = chat_tensor["input_ids"]
    else:
        input_ids = chat_tensor

    max_input_tokens = int(config["maxInputTokens"])
    if input_ids.shape[1] > max_input_tokens:
        input_ids = input_ids[:, -max_input_tokens:]

    input_ids = input_ids.to(model.device)

    primary_budget = int(max_new_tokens_override or config["maxNewTokensPrimary"])
    fallback_budget = int(config["maxNewTokensFallback"])
    max_tokens_used = primary_budget

    generation_start = time.time()
    try:
        with torch.no_grad():
            output_ids = model.generate(
                input_ids,
                max_new_tokens=primary_budget,
                min_new_tokens=8,
                pad_token_id=tokenizer.eos_token_id,
                **config["generationPrimary"],
            )
    except Exception as error:
        if "out of memory" not in str(error).lower():
            raise
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        max_tokens_used = fallback_budget
        with torch.no_grad():
            output_ids = model.generate(
                input_ids,
                max_new_tokens=fallback_budget,
                min_new_tokens=8,
                pad_token_id=tokenizer.eos_token_id,
                **config["generationFallback"],
            )

    generation_time_sec = round(time.time() - generation_start, 2)
    new_tokens = output_ids[0][input_ids.shape[1]:]

    decoded_raw = tokenizer.decode(new_tokens, skip_special_tokens=False)
    decoded_clean = tokenizer.decode(new_tokens, skip_special_tokens=True)

    token_count = int(len(new_tokens))
    tokens_per_sec = round(token_count / generation_time_sec, 2) if generation_time_sec > 0 else 0.0

    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "response": decoded_clean,
        "decodedRaw": decoded_raw,
        "decodedClean": decoded_clean,
        "responseCharCount": len(decoded_clean),
        "decodedRawCharCount": len(decoded_raw),
        "decodedCleanCharCount": len(decoded_clean),
        "tokenCount": token_count,
        "tokensPerSec": tokens_per_sec,
        "thinkingApplied": thinking_applied,
        "thinkingMode": thinking_mode,
        "thinkingFallbackReason": thinking_fallback_reason,
        "maxInputTokens": max_input_tokens,
        "maxNewTokensRequested": primary_budget,
        "maxNewTokensUsed": int(max_tokens_used),
        "quantizationApplied": quantization_applied,
        "timing": {
            "loadTimeSec": load_time_sec,
            "generationTimeSec": generation_time_sec,
        },
    }


def resolve_gguf_model_file(config: dict, cache_dir: Path, local_files_only: bool) -> Path:
    from huggingface_hub import snapshot_download

    allow_patterns = ["*.gguf"]
    gguf_file = config.get("ggufFile", "")
    if gguf_file:
        allow_patterns = [f"*{gguf_file}*.gguf", "*.gguf"]

    snapshot_path = snapshot_download(
        repo_id=config["executionModelId"],
        cache_dir=str(cache_dir),
        local_files_only=local_files_only,
        allow_patterns=allow_patterns,
    )

    snapshot_path = Path(snapshot_path)
    gguf_files = sorted(snapshot_path.rglob("*.gguf"))
    if not gguf_files:
        raise RuntimeError("No GGUF files found in snapshot")

    if gguf_file:
        filtered = [path for path in gguf_files if gguf_file.lower() in path.name.lower()]
        if filtered:
            return filtered[0]

    return gguf_files[0]


def run_llama_cpp_attempt(config: dict, cache_dir: Path, prompt_payload: dict, local_files_only: bool, disable_thinking: bool = False, max_new_tokens_override: int | None = None) -> dict:
    from llama_cpp import Llama

    load_start = time.time()
    gguf_path = resolve_gguf_model_file(config, cache_dir, local_files_only=local_files_only)

    prompt = build_prompt_for_non_template(
        prompt_payload["systemPrompt"],
        prompt_payload["userMessage"],
        thinking_requested=not disable_thinking,
    )

    max_input_chars = int(config["maxInputTokens"]) * 4
    if len(prompt) > max_input_chars:
        prompt = prompt[-max_input_chars:]

    runtime_kwargs = dict(config["modelKwargs"])
    llm = Llama(model_path=str(gguf_path), **runtime_kwargs)
    load_time_sec = round(time.time() - load_start, 2)

    primary_budget = int(max_new_tokens_override or config["maxNewTokensPrimary"])
    fallback_budget = int(config["maxNewTokensFallback"])
    max_tokens_used = primary_budget

    generation_start = time.time()
    try:
        completion = llm.create_completion(
            prompt=prompt,
            max_tokens=primary_budget,
            temperature=float(config["generationPrimary"].get("temperature", 0.7)),
            top_p=float(config["generationPrimary"].get("top_p", 0.9)),
        )
    except Exception:
        max_tokens_used = fallback_budget
        completion = llm.create_completion(
            prompt=prompt,
            max_tokens=fallback_budget,
            temperature=float(config["generationFallback"].get("temperature", 0.6)),
            top_p=float(config["generationFallback"].get("top_p", 0.85)),
        )

    generation_time_sec = round(time.time() - generation_start, 2)

    decoded_clean = completion["choices"][0]["text"]
    decoded_raw = decoded_clean

    usage = completion.get("usage", {})
    token_count = int(usage.get("completion_tokens", 0))
    if token_count <= 0:
        token_count = len(decoded_clean.split())

    tokens_per_sec = round(token_count / generation_time_sec, 2) if generation_time_sec > 0 else 0.0

    del llm
    gc.collect()

    return {
        "response": decoded_clean,
        "decodedRaw": decoded_raw,
        "decodedClean": decoded_clean,
        "responseCharCount": len(decoded_clean),
        "decodedRawCharCount": len(decoded_raw),
        "decodedCleanCharCount": len(decoded_clean),
        "tokenCount": token_count,
        "tokensPerSec": tokens_per_sec,
        "thinkingApplied": True,
        "thinkingMode": "prompt_enforced" if not disable_thinking else "prompt_non_thinking",
        "thinkingFallbackReason": "",
        "maxInputTokens": int(config["maxInputTokens"]),
        "maxNewTokensRequested": primary_budget,
        "maxNewTokensUsed": int(max_tokens_used),
        "quantizationApplied": True,
        "timing": {
            "loadTimeSec": load_time_sec,
            "generationTimeSec": generation_time_sec,
        },
    }


def classify_failure_phase(error_message: str) -> str:
    low = (error_message or "").lower()
    if "timeout" in low:
        return "timeout"
    if "out of memory" in low or ("cuda" in low and "memory" in low):
        return "generation"
    if "tokenizer" in low:
        return "tokenizer_load"
    if "401" in low or "gated" in low or "permission" in low:
        return "auth"
    if "model" in low:
        return "model_load"
    return "orchestration"


def run_model(payload: dict) -> dict:
    config = payload["config"]
    cache_dir = Path(payload["cacheDir"])
    cache_hit = bool(payload.get("cacheHit", False))
    prompt_payload = payload["promptPayload"]

    cache_dir.mkdir(parents=True, exist_ok=True)

    result = {
        "status": "failed",
        "failurePhase": "",
        "skipReason": "",
        "errorType": "",
        "errorMessage": "",
        "traceback": "",
        "cacheLoadMode": "",
        "cacheFallbackReason": "",
        "response": "",
        "responseCharCount": 0,
        "tokenCount": 0,
        "tokensPerSec": 0.0,
        "thinkingApplied": False,
        "thinkingMode": config.get("thinkingEnforcementMode", "template_flag"),
        "thinkingFallbackReason": "",
        "maxInputTokens": int(config["maxInputTokens"]),
        "maxNewTokensRequested": int(config["maxNewTokensPrimary"]),
        "maxNewTokensUsed": 0,
        "quantizationApplied": False,
        "timing": {
            "loadTimeSec": 0,
            "generationTimeSec": 0,
        },
        "decodedRawCharCount": 0,
        "decodedCleanCharCount": 0,
        "emptyVisibleOutput": False,
        "emptyOutputRetryTriggered": False,
        "emptyOutputRetrySucceeded": False,
        "emptyOutputRetryReason": "",
    }

    def run_backend(local_only: bool, disable_thinking: bool = False, max_new_tokens_override: int | None = None) -> dict:
        if config["backend"] == "transformers":
            return run_transformers_attempt(
                config=config,
                cache_dir=cache_dir,
                prompt_payload=prompt_payload,
                local_files_only=local_only,
                disable_thinking=disable_thinking,
                max_new_tokens_override=max_new_tokens_override,
            )
        return run_llama_cpp_attempt(
            config=config,
            cache_dir=cache_dir,
            prompt_payload=prompt_payload,
            local_files_only=local_only,
            disable_thinking=disable_thinking,
            max_new_tokens_override=max_new_tokens_override,
        )

    try:
        if cache_hit:
            try:
                inference = run_backend(local_only=True)
                result["cacheLoadMode"] = "local_only"
            except Exception as local_error:
                result["cacheFallbackReason"] = str(local_error)
                inference = run_backend(local_only=False)
                result["cacheLoadMode"] = "download"
        else:
            inference = run_backend(local_only=False)
            result["cacheLoadMode"] = "download"

        if inference["tokenCount"] > 0 and not (inference.get("decodedClean") or "").strip():
            result["emptyVisibleOutput"] = True
            result["emptyOutputRetryTriggered"] = True
            try:
                retry_tokens = max(64, int(config["maxNewTokensFallback"]))
                retry_inference = run_backend(local_only=True, disable_thinking=True, max_new_tokens_override=retry_tokens)
                result["emptyOutputRetrySucceeded"] = bool((retry_inference.get("decodedClean") or "").strip())
                if result["emptyOutputRetrySucceeded"]:
                    inference = retry_inference
                else:
                    result["emptyOutputRetryReason"] = "Retry completed but output remained empty"
            except Exception as retry_error:
                result["emptyOutputRetryReason"] = str(retry_error)

        result["status"] = "success"
        result["response"] = inference.get("decodedClean", "")
        result["responseCharCount"] = int(inference.get("responseCharCount", 0))
        result["tokenCount"] = int(inference.get("tokenCount", 0))
        result["tokensPerSec"] = float(inference.get("tokensPerSec", 0))
        result["thinkingApplied"] = bool(inference.get("thinkingApplied", False))
        result["thinkingMode"] = inference.get("thinkingMode", result["thinkingMode"])
        result["thinkingFallbackReason"] = inference.get("thinkingFallbackReason", "")
        result["maxInputTokens"] = int(inference.get("maxInputTokens", result["maxInputTokens"]))
        result["maxNewTokensRequested"] = int(inference.get("maxNewTokensRequested", result["maxNewTokensRequested"]))
        result["maxNewTokensUsed"] = int(inference.get("maxNewTokensUsed", 0))
        result["quantizationApplied"] = bool(inference.get("quantizationApplied", False))
        result["timing"] = inference.get("timing", result["timing"])
        result["decodedRawCharCount"] = int(inference.get("decodedRawCharCount", 0))
        result["decodedCleanCharCount"] = int(inference.get("decodedCleanCharCount", 0))

    except Exception as error:
        result["status"] = "failed"
        result["errorType"] = type(error).__name__
        result["errorMessage"] = str(error)
        result["traceback"] = traceback.format_exc()
        result["failurePhase"] = classify_failure_phase(str(error))
        result["skipReason"] = str(error)

    finally:
        gc.collect()
        try:
            import torch

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass

    return result


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Model worker")
    parser.add_argument("--payload", required=True)
    parser.add_argument("--output", required=True)
    return parser.parse_args()


def main() -> int:
    args = parse_args()
    payload_path = Path(args.payload)
    output_path = Path(args.output)

    try:
        with open(payload_path, "r", encoding="utf-8") as file_handle:
            payload = json.load(file_handle)
    except Exception as error:
        failure = {
            "status": "failed",
            "failurePhase": "bootstrap",
            "skipReason": str(error),
            "errorType": type(error).__name__,
            "errorMessage": str(error),
            "traceback": traceback.format_exc(),
        }
        atomic_write_json(output_path, failure)
        print(failure["traceback"], file=sys.stderr)
        return 1

    try:
        result = run_model(payload)
        atomic_write_json(output_path, result)
        if result.get("status") == "success":
            return 0
        print(result.get("traceback", result.get("errorMessage", "")), file=sys.stderr)
        return 1
    except Exception:
        failure = {
            "status": "failed",
            "failurePhase": "orchestration",
            "skipReason": "Unhandled worker exception",
            "errorType": "UnhandledWorkerException",
            "errorMessage": "Unhandled worker exception",
            "traceback": traceback.format_exc(),
        }
        atomic_write_json(output_path, failure)
        print(failure["traceback"], file=sys.stderr)
        return 1


if __name__ == "__main__":
    raise SystemExit(main())
"""


def getRunTimestamp():
    return datetime.now(timezone.utc).isoformat()


def buildPromptPayload():
    return {
        "systemPrompt": SYSTEM_PROMPT,
        "userMessage": USER_MESSAGE,
        "rubrics": RUBRICS,
    }


def getModelCacheDir(modelId):
    config = MODEL_CONFIGS[modelId]
    return NOTEBOOK_DIR / Path(config["cacheDir"])


def modelCacheExists(modelCacheDir):
    if not modelCacheDir.exists() or not modelCacheDir.is_dir():
        return False
    for path in modelCacheDir.rglob("*"):
        if path.is_file() and path.stat().st_size > 0:
            return True
    return False


def validateModelConfig(modelId):
    if modelId not in MODEL_CONFIGS:
        raise ValueError(f"Missing model config for {modelId}")
    config = MODEL_CONFIGS[modelId]
    missing = [key for key in REQUIRED_CONFIG_KEYS if key not in config]
    if missing:
        raise ValueError(f"Model config missing keys for {modelId}: {missing}")
    return config


def readJsonlRecords(path):
    if not path.exists():
        return []
    records = []
    with open(path, "r", encoding="utf-8") as fileHandle:
        for rawLine in fileHandle:
            line = rawLine.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except Exception:
                continue
    return records


def nextRunId():
    records = readJsonlRecords(RUN_LOG_JSONL_PATH)
    if not records:
        return 1
    return max(record.get("id", 0) for record in records) + 1


def atomicWriteJson(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile("w", delete=False, encoding="utf-8", dir=str(path.parent)) as tempFile:
        json.dump(payload, tempFile, ensure_ascii=True, indent=2)
        tempFile.flush()
        os.fsync(tempFile.fileno())
        tempPath = Path(tempFile.name)
    os.replace(tempPath, path)


def appendJsonl(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as fileHandle:
        fileHandle.write(json.dumps(payload, ensure_ascii=True) + "\n")
        fileHandle.flush()
        os.fsync(fileHandle.fileno())


def writeModelSnapshot(record):
    snapshotPath = BY_MODEL_DIR / f"{sanitizeModelId(record['modelId'])}.json"
    atomicWriteJson(snapshotPath, record)
    return snapshotPath


def evaluateWithGpt51(modelResponse, rubrics):
    judgePrompt = "You are a strict evaluation judge. Evaluate whether the given response satisfies the rubric. Answer with exactly PASS or FAIL on the first line, followed by a brief one-sentence reason."
    results = []
    start = time.time()
    client = OpenAI()

    for rubric in rubrics:
        try:
            response = client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[
                    {"role": "system", "content": judgePrompt},
                    {
                        "role": "user",
                        "content": f"Rubric:\n{rubric['description']}\n\nResponse:\n{modelResponse}",
                    },
                ],
            )
            raw = response.choices[0].message.content.strip()
            lines = [line.strip() for line in raw.splitlines() if line.strip()]
            first = lines[0].upper() if lines else "FAIL"
            passed = first.startswith("PASS")
            reason = lines[1] if len(lines) > 1 else (lines[0] if lines else "No explanation returned")
            results.append(
                {
                    "id": rubric["id"],
                    "rubric": rubric["description"],
                    "passed": passed,
                    "reasoning": reason,
                }
            )
        except Exception as e:
            results.append(
                {
                    "id": rubric["id"],
                    "rubric": rubric["description"],
                    "passed": False,
                    "reasoning": f"API Error: {str(e)}",
                }
            )

    elapsed = round(time.time() - start, 2)
    return results, elapsed


def classifyFailurePhase(errorMessage, stderrText):
    errorText = (errorMessage or "") + "\n" + (stderrText or "")
    low = errorText.lower()
    if "timeout" in low:
        return "timeout"
    if "tokenizer" in low:
        return "tokenizer_load"
    if "out of memory" in low or ("cuda" in low and "memory" in low):
        return "generation"
    if "401" in low or "gated" in low or "permission" in low:
        return "auth"
    if "model" in low:
        return "model_load"
    if "traceback" in low:
        return "orchestration"
    return "bootstrap"


def makeBaseRecord(runId, config):
    entry = MODEL_BY_ID[config["modelId"]]
    return {
        "id": runId,
        "modelId": config["modelId"],
        "targetModelId": config["targetModelId"],
        "executionModelId": config["executionModelId"],
        "backend": config["backend"],
        "quantizationMode": config["quantizationMode"],
        "surrogateUsed": config["surrogateUsed"],
        "surrogateReason": config["surrogateReason"],
        "family": entry["family"],
        "category": entry["category"],
        "params": entry["params"],
        "version": entry["version"],
        "timestampUtc": getRunTimestamp(),
        "runtimeProfile": RUNTIME_PROFILE,
        "hardware": HARDWARE_INFO,
        "status": "running",
        "prompt": buildPromptPayload(),
        "response": "",
        "responseCharCount": 0,
        "tokenCount": 0,
        "tokensPerSec": 0,
        "rubricDetails": [],
        "passCount": 0,
        "totalRubrics": len(RUBRICS),
        "passRate": 0,
        "timing": {
            "loadTimeSec": 0,
            "generationTimeSec": 0,
            "evaluationTimeSec": 0,
            "totalWallTimeSec": 0,
        },
        "configUsed": config,
        "thinkingRequested": bool(config["thinkingRequested"]),
        "thinkingApplied": False,
        "thinkingMode": config["thinkingEnforcementMode"],
        "thinkingFallbackReason": "",
        "maxInputTokens": int(config["maxInputTokens"]),
        "maxNewTokensRequested": int(config["maxNewTokensPrimary"]),
        "maxNewTokensUsed": 0,
        "failurePhase": "",
        "skipReason": "",
        "errorType": "",
        "errorMessage": "",
        "traceback": "",
        "cacheCheckPerformed": True,
        "cacheHit": False,
        "cacheLoadMode": "",
        "cacheFallbackReason": "",
        "cacheCleanupSucceeded": None,
        "cacheCleanupError": "",
        "quantizationRequested": config["quantizationMode"] in ["bnb4", "gguf"],
        "quantizationApplied": False,
        "workerCrashed": False,
        "workerTimeout": False,
        "workerReturnCode": None,
        "workerStdout": "",
        "workerStderr": "",
        "workerCommand": "",
        "workerDurationSec": 0,
        "workerOutputJsonExists": False,
        "workerFailureStage": "",
        "decodedRawCharCount": 0,
        "decodedCleanCharCount": 0,
        "emptyVisibleOutput": False,
        "emptyOutputRetryTriggered": False,
        "emptyOutputRetrySucceeded": False,
        "emptyOutputRetryReason": "",
    }


def runWorker(payloadPathRel, outputPathRel, timeoutSec):
    command = [
        sys.executable,
        "-",
        "--payload",
        payloadPathRel.as_posix(),
        "--output",
        outputPathRel.as_posix(),
    ]
    started = time.time()
    completed = subprocess.run(
        command,
        input=MODEL_WORKER_INLINE_CODE,
        capture_output=True,
        text=True,
        timeout=int(timeoutSec),
        cwd=str(NOTEBOOK_DIR),
    )
    durationSec = round(time.time() - started, 2)
    return command, completed, durationSec


def runAndEvaluateModel(modelId):
    modelStart = time.time()
    runId = nextRunId()
    config = validateModelConfig(modelId)
    record = makeBaseRecord(runId, config)

    modelCacheDir = getModelCacheDir(modelId)
    modelCacheDir.mkdir(parents=True, exist_ok=True)
    hasCache = modelCacheExists(modelCacheDir)
    record["cacheHit"] = hasCache

    print("\n" + "=" * 100)
    print(f"RUN ID: {runId}")
    print(f"TARGET MODEL: {record['targetModelId']}")
    print(f"EXECUTION MODEL: {record['executionModelId']}")
    print(f"FAMILY: {record['family']} | CATEGORY: {record['category']} | PARAMS: {record['params']}")
    print(f"Backend: {record['backend']} | Quantization: {record['quantizationMode']}")
    print(f"Surrogate used: {record['surrogateUsed']}")
    if record["surrogateUsed"]:
        print(f"Surrogate reason: {record['surrogateReason']}")
    print(f"Cache dir (relative): {Path(config['cacheDir']).as_posix()}")
    print(f"Cache hit: {hasCache}")
    print("=" * 100)

    writeModelSnapshot(record)

    tmpDir = RESULTS_DIR / "tmp"
    tmpDir.mkdir(parents=True, exist_ok=True)
    payloadPath = tmpDir / f"{sanitizeModelId(modelId)}__{runId}__payload.json"
    outputPath = tmpDir / f"{sanitizeModelId(modelId)}__{runId}__output.json"

    payload = {
        "config": config,
        "cacheDir": str(modelCacheDir),
        "cacheHit": hasCache,
        "promptPayload": {
            "systemPrompt": SYSTEM_PROMPT,
            "userMessage": USER_MESSAGE,
        },
        "hardwareInfo": HARDWARE_INFO,
    }

    atomicWriteJson(payloadPath, payload)

    payloadPathRel = payloadPath.relative_to(NOTEBOOK_DIR)
    outputPathRel = outputPath.relative_to(NOTEBOOK_DIR)

    workerResult = None

    try:
        command, completed, durationSec = runWorker(payloadPathRel, outputPathRel, config["workerTimeoutSec"])
        record["workerCommand"] = " ".join(command)
        record["workerDurationSec"] = durationSec
        record["workerReturnCode"] = completed.returncode
        record["workerStdout"] = completed.stdout
        record["workerStderr"] = completed.stderr
        record["workerOutputJsonExists"] = outputPath.exists()

        if outputPath.exists():
            try:
                with open(outputPath, "r", encoding="utf-8") as fileHandle:
                    workerResult = json.load(fileHandle)
            except Exception as e:
                record["status"] = "failed"
                record["failurePhase"] = "bootstrap"
                record["workerFailureStage"] = "bootstrap"
                record["skipReason"] = f"Worker output parse failed: {str(e)}"
                record["errorType"] = type(e).__name__
                record["errorMessage"] = str(e)
                record["traceback"] = traceback.format_exc()

        if completed.returncode != 0 and workerResult is None:
            record["status"] = "failed"
            record["workerCrashed"] = True
            record["failurePhase"] = "worker_crash"
            record["workerFailureStage"] = classifyFailurePhase(record["errorMessage"], record["workerStderr"])
            record["skipReason"] = "Worker exited with non-zero code before result payload"
            record["errorType"] = "WorkerCrash"
            record["errorMessage"] = f"exitcode={completed.returncode}"

    except subprocess.TimeoutExpired as e:
        record["workerDurationSec"] = round(float(e.timeout), 2)
        record["workerTimeout"] = True
        record["status"] = "skipped"
        record["failurePhase"] = "timeout"
        record["workerFailureStage"] = "timeout"
        record["skipReason"] = f"Worker timeout after {config['workerTimeoutSec']} seconds"
        record["errorType"] = "TimeoutExpired"
        record["errorMessage"] = str(e)
        record["workerStdout"] = (e.stdout or "") if hasattr(e, "stdout") else ""
        record["workerStderr"] = (e.stderr or "") if hasattr(e, "stderr") else ""

    except Exception as e:
        record["status"] = "failed"
        record["failurePhase"] = "orchestration"
        record["workerFailureStage"] = "orchestration"
        record["skipReason"] = str(e)
        record["errorType"] = type(e).__name__
        record["errorMessage"] = str(e)
        record["traceback"] = traceback.format_exc()

    try:
        if workerResult is not None:
            record["status"] = workerResult.get("status", record["status"])
            record["failurePhase"] = workerResult.get("failurePhase", record["failurePhase"])
            record["skipReason"] = workerResult.get("skipReason", record["skipReason"])
            record["errorType"] = workerResult.get("errorType", record["errorType"])
            record["errorMessage"] = workerResult.get("errorMessage", record["errorMessage"])
            record["traceback"] = workerResult.get("traceback", record["traceback"])
            record["cacheLoadMode"] = workerResult.get("cacheLoadMode", "")
            record["cacheFallbackReason"] = workerResult.get("cacheFallbackReason", "")
            record["response"] = workerResult.get("response", "")
            record["responseCharCount"] = int(workerResult.get("responseCharCount", 0))
            record["tokenCount"] = int(workerResult.get("tokenCount", 0))
            record["tokensPerSec"] = float(workerResult.get("tokensPerSec", 0))
            record["thinkingApplied"] = bool(workerResult.get("thinkingApplied", False))
            record["thinkingMode"] = workerResult.get("thinkingMode", record["thinkingMode"])
            record["thinkingFallbackReason"] = workerResult.get("thinkingFallbackReason", "")
            record["maxInputTokens"] = int(workerResult.get("maxInputTokens", record["maxInputTokens"]))
            record["maxNewTokensRequested"] = int(workerResult.get("maxNewTokensRequested", record["maxNewTokensRequested"]))
            record["maxNewTokensUsed"] = int(workerResult.get("maxNewTokensUsed", 0))
            record["quantizationApplied"] = bool(workerResult.get("quantizationApplied", False))
            record["decodedRawCharCount"] = int(workerResult.get("decodedRawCharCount", 0))
            record["decodedCleanCharCount"] = int(workerResult.get("decodedCleanCharCount", 0))
            record["emptyVisibleOutput"] = bool(workerResult.get("emptyVisibleOutput", False))
            record["emptyOutputRetryTriggered"] = bool(workerResult.get("emptyOutputRetryTriggered", False))
            record["emptyOutputRetrySucceeded"] = bool(workerResult.get("emptyOutputRetrySucceeded", False))
            record["emptyOutputRetryReason"] = workerResult.get("emptyOutputRetryReason", "")
            timing = workerResult.get("timing", {})
            record["timing"]["loadTimeSec"] = float(timing.get("loadTimeSec", 0))
            record["timing"]["generationTimeSec"] = float(timing.get("generationTimeSec", 0))

            if record["status"] == "failed" and not record["workerFailureStage"]:
                record["workerFailureStage"] = workerResult.get("failurePhase", "")

            writeModelSnapshot(record)

            if record["status"] == "success":
                rubricDetails, evalTime = evaluateWithGpt51(record["response"], RUBRICS)
                passCount = sum(1 for detail in rubricDetails if detail["passed"])
                passRate = round(passCount / len(RUBRICS) * 100, 1)
                record["rubricDetails"] = rubricDetails
                record["passCount"] = passCount
                record["passRate"] = passRate
                record["timing"]["evaluationTimeSec"] = evalTime

    finally:
        gc.collect()
        if DEVICE == "mps":
            torch.mps.empty_cache()
        elif DEVICE == "cuda":
            torch.cuda.empty_cache()

        record["timing"]["totalWallTimeSec"] = round(time.time() - modelStart, 2)

        writeModelSnapshot(record)
        appendJsonl(RUN_LOG_JSONL_PATH, record)
        RESULTS_BY_MODEL[modelId] = record

    if record["response"]:
        print("\nMODEL OUTPUT:")
        print(record["response"])

    if record["rubricDetails"]:
        print("\nRUBRIC EVALUATION:")
        for detail in record["rubricDetails"]:
            status = "PASS" if detail["passed"] else "FAIL"
            print(f"{detail['id']} {status}")
            print(f"Rubric: {detail['rubric']}")
            print(f"Judge: {detail['reasoning']}")
            print("-" * 80)
        print(f"Final Score: {record['passCount']}/{record['totalRubrics']} ({record['passRate']}%)")

    if record["status"] != "success":
        print(f"FAILED OR SKIPPED: {record['skipReason']}")
        if record["traceback"]:
            print(record["traceback"])
        if record["workerStderr"]:
            print("\nWORKER STDERR:")
            print(record["workerStderr"])

    return record


def mergeResults():
    records = []
    perModelFiles = sorted(BY_MODEL_DIR.glob("*.json"))

    if perModelFiles:
        for filePath in perModelFiles:
            try:
                with open(filePath, "r", encoding="utf-8") as fileHandle:
                    records.append(json.load(fileHandle))
            except Exception:
                continue
    else:
        records = readJsonlRecords(RUN_LOG_JSONL_PATH)

    latestByModel = {}
    for record in records:
        modelId = record.get("modelId")
        if not modelId:
            continue
        timestamp = record.get("timestampUtc", "")
        current = latestByModel.get(modelId)
        if current is None or timestamp >= current.get("timestampUtc", ""):
            latestByModel[modelId] = record

    mergedRecords = [latestByModel[entry["modelId"]] for entry in MODEL_REGISTRY if entry["modelId"] in latestByModel]
    atomicWriteJson(MERGED_RESULTS_PATH, mergedRecords)

    missingModelIds = [entry["modelId"] for entry in MODEL_REGISTRY if entry["modelId"] not in latestByModel]
    staleModelIds = [modelId for modelId in latestByModel if modelId not in MODEL_BY_ID]

    successCount = sum(1 for record in mergedRecords if record.get("status") == "success")
    failedOrSkippedCount = sum(1 for record in mergedRecords if record.get("status") != "success")

    print("Merge summary:")
    print(f"  success count: {successCount}")
    print(f"  failed or skipped count: {failedOrSkippedCount}")
    print(f"  missing model IDs: {missingModelIds}")
    print(f"  stale model IDs: {staleModelIds}")

    evalDf = pd.DataFrame(
        [
            {
                "Model": record["modelId"].split("/")[-1],
                "Model ID": record["modelId"],
                "Execution Model ID": record.get("executionModelId", ""),
                "Surrogate": record.get("surrogateUsed", False),
                "Status": record.get("status", ""),
                "Pass Rate (%)": record.get("passRate", 0),
                "Pass": record.get("passCount", 0),
                "Total": record.get("totalRubrics", len(RUBRICS)),
                "Time Spent (s)": record.get("timing", {}).get("totalWallTimeSec", 0),
                "Speed (tok/s)": record.get("tokensPerSec", 0),
                "Failure Phase": record.get("failurePhase", ""),
                "Skip Reason": record.get("skipReason", ""),
                "Error Type": record.get("errorType", ""),
                "Worker RC": record.get("workerReturnCode", None),
                "Worker Stage": record.get("workerFailureStage", ""),
                "Cache Hit": record.get("cacheHit", False),
                "Cache Load Mode": record.get("cacheLoadMode", ""),
            }
            for record in mergedRecords
        ]
    )

    if not evalDf.empty:
        evalDf = evalDf.sort_values(["Pass Rate (%)", "Speed (tok/s)"], ascending=[False, False]).reset_index(drop=True)

    return mergedRecords, evalDf



In [ ]:

def clearModelCache(cacheDir):
    cleanupError = ""
    succeeded = True
    try:
        if cacheDir.exists():
            shutil.rmtree(cacheDir)
        cacheDir.mkdir(parents=True, exist_ok=True)
    except Exception as e:
        succeeded = False
        cleanupError = str(e)
    return succeeded, cleanupError


def clearModelCacheForModel(modelId: str) -> dict:
    if modelId not in MODEL_CONFIGS:
        raise ValueError(f"Unknown model ID: {modelId}")
    cacheDir = NOTEBOOK_DIR / Path(MODEL_CONFIGS[modelId]["cacheDir"])
    succeeded, errorMessage = clearModelCache(cacheDir)
    result = {
        "modelId": modelId,
        "cacheDir": str(Path(MODEL_CONFIGS[modelId]["cacheDir"])),
        "succeeded": succeeded,
        "error": errorMessage,
    }
    print(result)
    return result


def clearAllModelCache() -> dict:
    cleared = []
    errors = []
    for modelId in MODEL_CONFIGS:
        cacheDir = NOTEBOOK_DIR / Path(MODEL_CONFIGS[modelId]["cacheDir"])
        succeeded, errorMessage = clearModelCache(cacheDir)
        if succeeded:
            cleared.append(modelId)
        else:
            errors.append({"modelId": modelId, "error": errorMessage})
    result = {
        "clearedModelCount": len(cleared),
        "errorCount": len(errors),
        "errors": errors,
    }
    print(result)
    return result


In [ ]:
runAndEvaluateModel("Qwen/Qwen3.5-0.8B")

In [ ]:
runAndEvaluateModel("Qwen/Qwen3.5-2B")

In [ ]:
runAndEvaluateModel("Qwen/Qwen3.5-4B")

In [ ]:
runAndEvaluateModel("Qwen/Qwen3.5-9B")

In [ ]:
runAndEvaluateModel("Qwen/Qwen3.5-27B")

In [ ]:
runAndEvaluateModel("google/gemma-3-1b-it")

In [ ]:
runAndEvaluateModel("google/gemma-2-2b-it")

In [ ]:
runAndEvaluateModel("google/gemma-3-4b-it")

In [ ]:
runAndEvaluateModel("google/gemma-3-12b-it")

In [ ]:
runAndEvaluateModel("google/gemma-3-27b-it")

In [ ]:
runAndEvaluateModel("zai-org/glm-edge-1.5b-chat")

In [ ]:
runAndEvaluateModel("zai-org/glm-edge-4b-chat")

In [ ]:
runAndEvaluateModel("zai-org/GLM-4-9B-0414")

In [ ]:
runAndEvaluateModel("zai-org/GLM-4.7-Flash")

In [ ]:
runAndEvaluateModel("zai-org/GLM-4.5-Air")

---
## Merge All Model Results

In [ ]:
MERGED_RESULTS, evalDf = mergeResults()
evalDf

---
## Reporting and Accuracy vs Time Plot

This section builds the reporting dataframe from merged JSON and plots model accuracy against total time spent.

In [ ]:
if not MERGED_RESULTS_PATH.exists():
    raise RuntimeError("Merged results not found. Run the merge cell first.")

with open(MERGED_RESULTS_PATH, "r", encoding="utf-8") as fileHandle:
    mergedRecords = json.load(fileHandle)

reportDf = pd.DataFrame(
    [
        {
            "Model ID": record.get("modelId", ""),
            "Execution Model ID": record.get("executionModelId", ""),
            "Surrogate": record.get("surrogateUsed", False),
            "Status": record.get("status", ""),
            "Pass Rate (%)": record.get("passRate", 0),
            "Time Spent (s)": record.get("timing", {}).get("totalWallTimeSec", 0),
            "Worker RC": record.get("workerReturnCode", None),
            "Worker Stage": record.get("workerFailureStage", ""),
            "Error Type": record.get("errorType", ""),
            "Skip Reason": record.get("skipReason", ""),
        }
        for record in mergedRecords
    ]
)

if reportDf.empty:
    print("No merged records found.")
else:
    reportDf = reportDf.sort_values(["Pass Rate (%)", "Time Spent (s)"], ascending=[False, True]).reset_index(drop=True)
    reportDf


In [ ]:

if reportDf.empty:
    print("No data to plot.")
else:
    plt.figure(figsize=(12, 7))

    base = reportDf[reportDf["Surrogate"] == False]
    surrogate = reportDf[reportDf["Surrogate"] == True]

    plt.scatter(base["Pass Rate (%)"], base["Time Spent (s)"], label="Exact execution", marker="o", s=100)
    plt.scatter(surrogate["Pass Rate (%)"], surrogate["Time Spent (s)"], label="Surrogate execution", marker="^", s=120)

    for _, row in reportDf.iterrows():
        label = row["Model ID"].split("/")[-1]
        plt.annotate(label, (row["Pass Rate (%)"], row["Time Spent (s)"]), fontsize=8, xytext=(4, 4), textcoords="offset points")

    plt.title("Model Accuracy vs Time Spent")
    plt.xlabel("Pass Rate (%)")
    plt.ylabel("Total Time Spent (s)")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()
